# Support Ticket Auto-Tagging with LLMs — Zero-shot vs Few-shot

**Task 5 — AI/ML Engineering Advanced Internship, DevelopersHub Corporation**

**Objective:** Automatically classify free-text customer support tickets into
categories using LLM techniques, compare zero-shot vs few-shot prompting, and
output the top-3 most probable categories per ticket with confidence scores.

This notebook walks through the full pipeline: dataset loading & preprocessing,
zero-shot classification, few-shot classification, evaluation, visualization,
and final conclusions. It calls into the reusable modules under `src/` so the
same logic backs both this notebook and the Streamlit app.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src...` works when running from notebooks/

import pandas as pd
import json
from src.config import CATEGORIES, RESULTS_DIR
from src.preprocessing import load_dataset, split_dataset

pd.set_option("display.max_colwidth", 120)


## 2. Dataset loading & preprocessing

In [ ]:
df = load_dataset()
print(f"Total tickets: {len(df)}")
print(f"Categories: {df['category'].nunique()}")
df['category'].value_counts()


In [ ]:
train_df, test_df = split_dataset(df)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
test_df.head(10)


## 3. Zero-shot classification

Using `facebook/bart-large-mnli` via HuggingFace's `zero-shot-classification`
pipeline. Each candidate category becomes an NLI hypothesis and is scored by
entailment probability — no training examples required.

In [ ]:
from src import zero_shot_classifier as zsc

sample_ticket = test_df.iloc[0]["text"]
print("Ticket:", sample_ticket)
print("True category:", test_df.iloc[0]["category"])
print()
for item in zsc.classify_ticket(sample_ticket, top_k=3):
    print(f"{item['label']:30s} {item['score']:.3f}")


## 4. Few-shot classification

Using `google/flan-t5-base` with 1-2 in-context examples per category. Instead
of free-form generation, we score each candidate category's log-likelihood as
a forced decoder continuation (length-normalized, then softmaxed) — this gives
a clean, ranked confidence distribution comparable to the zero-shot output.

In [ ]:
from src import few_shot_classifier as fsc

print("Ticket:", sample_ticket)
print("True category:", test_df.iloc[0]["category"])
print()
for item in fsc.classify_ticket(sample_ticket, shots_per_category=1, top_k=3):
    print(f"{item['label']:30s} {item['score']:.3f}")


## 5. Full evaluation: zero-shot vs few-shot

Runs both classifiers over the entire test split and computes accuracy,
macro precision/recall/F1, and top-3 hit rate for each. Results are saved to
`results/metrics.json` and `results/predictions.csv`.

> This step downloads ~1.6GB of model weights and runs inference on every
> test ticket with both methods — it can take several minutes on CPU. Use
> `run_full_evaluation(n_test_samples=20)` for a quick smoke test.

In [ ]:
from src.evaluate import run_full_evaluation

zs_metrics, fs_metrics, predictions_df = run_full_evaluation(shots_per_category=1)


In [ ]:
predictions_df.head(10)


## 6. Visualizations

In [ ]:
from src.visualize import generate_all_plots
generate_all_plots()


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULTS_DIR / "figures" / "metric_comparison.png")))


In [ ]:
display(Image(filename=str(RESULTS_DIR / "figures" / "confusion_zero_shot.png")))
display(Image(filename=str(RESULTS_DIR / "figures" / "confusion_few_shot.png")))


In [ ]:
display(Image(filename=str(RESULTS_DIR / "figures" / "per_category_f1.png")))
display(Image(filename=str(RESULTS_DIR / "figures" / "ranking_breakdown.png")))


## 7. Summary & Insights

- **Zero-shot** (`bart-large-mnli`) needs no labeled examples and scores
  candidate categories via NLI entailment — a strong baseline when category
  names are self-descriptive.
- **Few-shot** (`flan-t5-base` + log-likelihood scoring) uses a handful of
  in-context examples per category and tends to help most on categories with
  overlapping vocabulary (e.g. Cancellation Request vs. Billing and Payments).
- **Top-3 accuracy** is substantially higher than top-1 for both methods —
  most misclassifications are near-misses between semantically adjacent
  categories, which is exactly why returning the top-3 (not just top-1) is
  valuable for a real ticket-routing assistant: a human agent can pick from
  a short, high-recall shortlist instead of a single guess.
- **Latency tradeoff:** zero-shot classifies all categories in one batched
  NLI pass; the few-shot scorer used here needs one forward pass per
  candidate category, which matters for production throughput.

See `report.md` at the project root for the full write-up, and
`README.md` for setup and usage instructions.